<a href="https://colab.research.google.com/github/HoangTimothy/DL-Assignment-DNAC1/blob/main/notebooks/DL_BTL1_Multimodal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
data_path = kagglehub.dataset_download("wenewone/cub2002011")

print("Path to dataset files:", data_path)

Using Colab cache for faster access to the 'cub2002011' dataset.
Path to dataset files: /kaggle/input/cub2002011


In [2]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
import random

class CUBMultimodalDataset(Dataset):
    def __init__(self, root_dir, split='train', processor=None, k_shots=None, seed=42):
        """
        root_dir: Thư mục gốc chứa tập dữ liệu CUB (chứa thư mục images và text)
        split: 'train' hoặc 'test'
        processor: CLIPProcessor từ Hugging Face
        k_shots: Số lượng mẫu mỗi lớp dành cho kịch bản Few-shot. Nếu None, dùng toàn bộ.
        """
        self.root_dir = root_dir
        image_root_dir = os.path.join(root_dir, 'CUB_200_2011')
        self.split = split
        self.processor = processor

        # Đường dẫn đến các tệp siêu dữ liệu chuẩn của CUB
        images_txt = os.path.join(image_root_dir, 'images.txt')
        split_txt = os.path.join(image_root_dir, 'train_test_split.txt')
        labels_txt = os.path.join(image_root_dir, 'image_class_labels.txt')

        # Đọc dữ liệu thành bảng
        images_df = pd.read_csv(images_txt, sep=' ', names=['img_id', 'filepath'])
        split_df = pd.read_csv(split_txt, sep=' ', names=['img_id', 'is_train'])
        labels_df = pd.read_csv(labels_txt, sep=' ', names=['img_id', 'class_id'])

        # Gộp các bảng lại bằng img_id
        df = images_df.merge(split_df, on='img_id').merge(labels_df, on='img_id')

        # Lọc theo tập huấn luyện hoặc kiểm thử
        is_train_flag = 1 if split == 'train' else 0
        self.data = df[df['is_train'] == is_train_flag].reset_index(drop=True)

        # Xử lý kịch bản Few-shot: Lấy ngẫu nhiên k mẫu cho mỗi lớp
        if split == 'train' and k_shots is not None:
            # Trộn ngẫu nhiên trước khi gom nhóm để đảm bảo tính khách quan
            self.data = self.data.sample(frac=1, random_state=seed).reset_index(drop=True)
            self.data = self.data.groupby('class_id').head(k_shots).reset_index(drop=True)

        self.image_dir = os.path.join(data_path, 'CUB_200_2011', 'images')
        self.text_dir = os.path.join(root_dir, 'cvpr2016_cub', 'text_c10')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filepath'])

        # Đường dẫn tệp văn bản thường có cùng cấu trúc thư mục nhưng đuôi .txt
        text_relative_path = row['filepath'].replace('.jpg', '.txt')
        text_path = os.path.join(self.text_dir, text_relative_path)

        # Nạp hình ảnh (chuyển về RGB để tránh lỗi với ảnh thang độ xám)
        image = Image.open(img_path).convert('RGB')

        # Nạp toàn bộ các câu miêu tả (thường có 10 câu cho mỗi ảnh)
        with open(text_path, 'r', encoding='utf-8') as f:
            captions = f.read().splitlines()

        # Tiền xử lý văn bản: Xóa các dòng rỗng
        captions = [c.strip() for c in captions if c.strip() != '']

        # Tăng cường dữ liệu văn bản: Chọn ngẫu nhiên 1 câu khi huấn luyện
        if self.split == 'train':
            text = random.choice(captions)
        else:
            text = captions[0] # Giữ cố định câu đầu tiên khi kiểm thử để đánh giá ổn định

        # Nhãn gốc của CUB bắt đầu từ 1, đưa về 0 để tương thích với PyTorch
        label = row['class_id'] - 1

        if self.processor:
            # Bộ xử lý CLIP sẽ tự động chuẩn hóa ảnh và mã hóa văn bản
            encoding = self.processor(
                text=text,
                images=image,
                return_tensors="pt",
                padding="max_length",
                truncation=True,
                max_length=77 # Độ dài chuỗi tối đa chuẩn của CLIP
            )
            # Loại bỏ chiều lô dữ liệu giả do bộ xử lý sinh ra
            encoding = {k: v.squeeze(0) for k, v in encoding.items()}
            encoding['label'] = torch.tensor(label, dtype=torch.long)
            return encoding

        return image, text, label

In [3]:
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import DataLoader

# Khởi tạo mô hình và bộ xử lý CLIP cơ sở
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id, use_fast=False)

# Tạo tập dữ liệu Few-shot (ví dụ 5 mẫu mỗi lớp)
train_dataset = CUBMultimodalDataset(
    root_dir=data_path,
    split='train',
    processor=processor,
    k_shots=5
)

# Nạp vào bộ Dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Lấy thử một lô dữ liệu để kiểm tra hình dạng tensor
batch = next(iter(train_loader))
print("Kích thước tensor hình ảnh:", batch['pixel_values'].shape)
print("Kích thước tensor văn bản:", batch['input_ids'].shape)
print("Kích thước tensor nhãn:", batch['label'].shape)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Kích thước tensor hình ảnh: torch.Size([32, 3, 224, 224])
Kích thước tensor văn bản: torch.Size([32, 77])
Kích thước tensor nhãn: torch.Size([32])


In [4]:
import os
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

def get_cub_class_names(root_dir):
    """Đọc và làm sạch tên 200 loài chim từ tệp classes.txt"""
    classes_txt = os.path.join(root_dir, 'CUB_200_2011', 'classes.txt')
    # Cấu trúc tệp: "1 001.Black_footed_Albatross"
    classes_df = pd.read_csv(classes_txt, sep=' ', names=['class_id', 'class_name'])

    # Loại bỏ tiền tố số và thay dấu gạch dưới bằng khoảng trắng
    # Kết quả: "Black footed Albatross"
    clean_names = classes_df['class_name'].apply(lambda x: x.split('.')[1].replace('_', ' '))
    return clean_names.tolist()

# Zero-Shot

In [5]:
def evaluate_zero_shot_clip(model, processor, test_loader, class_names, device):
    model.eval()
    model.to(device)

    # 1. Kỹ thuật Prompt Ensembling (Tổ hợp câu lệnh)
    templates = [
        "a photo of a {}, a type of bird.",
        "a cropped photo of a {}, a bird.",
        "a close-up photo of a {}, a bird.",
        "a photo of the {}, a bird.",
        "this is a photo of a {}, a type of bird."
    ]

    print("Đang mã hóa văn bản bằng Prompt Ensembling...")
    text_features_list = []

    with torch.no_grad():
        for name in tqdm(class_names, desc="Text Encoding"):
            # Tạo 5 câu lệnh cho 1 loài chim
            prompts = [template.format(name) for template in templates]
            inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(device)

            # Trích xuất vector an toàn (truy cập thẳng vào mạng lõi)
            text_outputs = model.text_model(**inputs)
            text_embeds = text_outputs[1] # Lấy pooler_output
            text_embeds = model.text_projection(text_embeds)

            # Chuẩn hóa từng câu lệnh
            text_embeds = text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)

            # Lấy trung bình cộng vector của 5 câu lệnh
            mean_emb = text_embeds.mean(dim=0)
            # Chuẩn hóa vector trung bình một lần nữa
            mean_emb = mean_emb / mean_emb.norm(p=2, dim=-1)

            text_features_list.append(mean_emb)

    # Ghép lại thành ma trận chuẩn (200 lớp, 512 chiều)
    text_features = torch.stack(text_features_list)

    all_preds = []
    all_labels = []

    # 2. Lấy tham số Logit Scale chuẩn của CLIP
    logit_scale = model.logit_scale.exp().item()

    print("Đang chạy dự đoán Zero-shot...")
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Zero-shot Inference"):
            images = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)

            # Trích xuất vector ảnh an toàn
            vision_outputs = model.vision_model(images)
            image_embeds = vision_outputs[1]
            image_embeds = model.visual_projection(image_embeds)

            image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)

            # 3. Tính toán logits chuẩn mực
            logits = logit_scale * image_embeds @ text_features.T

            # Chọn nhãn dự đoán
            _, preds = logits.max(dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 4. Đánh giá
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    print("\n" + "="*40)
    print("KẾT QUẢ ZERO-SHOT ĐÃ ĐƯỢC TỐI ƯU")
    print("="*40)
    print(f"Độ chính xác (Accuracy): {acc*100:.2f}%")
    print(f"Điểm F1 Macro: {f1*100:.2f}%")

    return acc, f1, all_labels, all_preds

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import DataLoader

# Khởi tạo thiết bị và mô hình
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch32"

print("Đang tải mô hình CLIP pretrained...")
processor = CLIPProcessor.from_pretrained(model_id)
clip_model = CLIPModel.from_pretrained(model_id)

# Lấy danh sách tên 200 loài chim
cub_root_dir = data_path
bird_class_names = get_cub_class_names(cub_root_dir)

# Khởi tạo tập kiểm thử (test split)
# Ở Zero-shot, chúng ta không cần truyền k_shots
test_dataset = CUBMultimodalDataset(
    root_dir=cub_root_dir,
    split='test',
    processor=processor
)

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Chạy đánh giá
zero_shot_acc, zero_shot_f1, true_labels, pred_labels = evaluate_zero_shot_clip(
    model=clip_model,
    processor=processor,
    test_loader=test_loader,
    class_names=bird_class_names,
    device=device
)

Đang tải mô hình CLIP pretrained...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đang mã hóa văn bản bằng Prompt Ensembling...


Text Encoding: 100%|██████████| 200/200 [00:04<00:00, 43.95it/s]


Đang chạy dự đoán Zero-shot...


Zero-shot Inference:  81%|████████▏ | 74/91 [01:02<00:11,  1.53it/s]

# Few-Shot

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset
from tqdm import tqdm

def extract_multimodal_features(loader, model, device):
    model.eval()
    model.to(device)

    # Đưa hàm vá lỗi thông minh vào lại đây để dùng chung
    def extract_clip_features(raw_features, projection_layer):
        if isinstance(raw_features, torch.Tensor):
            return raw_features
        if hasattr(raw_features, 'text_embeds') and raw_features.text_embeds is not None:
            return raw_features.text_embeds
        if hasattr(raw_features, 'image_embeds') and raw_features.image_embeds is not None:
            return raw_features.image_embeds
        if hasattr(raw_features, 'pooler_output'):
            out = raw_features.pooler_output
            if projection_layer is not None and out.shape[-1] == projection_layer.in_features:
                out = projection_layer(out)
            return out
        if isinstance(raw_features, tuple):
            return raw_features[0]
        return raw_features

    features_list = []
    labels_list = []

    print("Đang trích xuất đặc trưng đa phương thức...")
    with torch.no_grad():
        for batch in tqdm(loader, desc="Feature Extraction"):
            images = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # 1. Trích xuất đặc trưng thô
            raw_img = model.get_image_features(pixel_values=images)
            raw_txt = model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)

            # 2. Sử dụng hàm vá lỗi để ép kiểu lấy Tensor chuẩn (512 chiều)
            img_features = extract_clip_features(raw_img, model.visual_projection)
            txt_features = extract_clip_features(raw_txt, model.text_projection)

            # 3. Chuẩn hóa vector độ dài về 1
            img_features = img_features / img_features.norm(p=2, dim=-1, keepdim=True)
            txt_features = txt_features / txt_features.norm(p=2, dim=-1, keepdim=True)

            # 4. Nối vector hình ảnh và văn bản thành vector 1024 chiều
            multimodal_features = torch.cat([img_features, txt_features], dim=1)

            features_list.append(multimodal_features.cpu())
            labels_list.append(labels.cpu())

    return torch.cat(features_list), torch.cat(labels_list)

In [ ]:
class MultimodalClassifier(nn.Module):
    def __init__(self, input_dim=1024, num_classes=200):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

In [ ]:
# 1. Khởi tạo tập dữ liệu Few-shot (ví dụ k=10)
few_shot_train_dataset = CUBMultimodalDataset(
    root_dir=cub_root_dir,
    split='train',
    processor=processor,
    k_shots=10
)
few_shot_train_loader = DataLoader(few_shot_train_dataset, batch_size=64, shuffle=True)

# 2. Trích xuất đặc trưng cho cả tập huấn luyện và kiểm thử
print("Xử lý tập Few-shot Train...")
X_train, y_train = extract_multimodal_features(few_shot_train_loader, clip_model, device)

print("Xử lý tập Test...")
X_test, y_test = extract_multimodal_features(test_loader, clip_model, device)

# 3. Chuyển đổi thành TensorDataset nạp vào PyTorch
train_features_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

# 4. Huấn luyện mạng tuyến tính
classifier = MultimodalClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier.parameters(), lr=1e-3)

epochs = 50
print("\nBắt đầu huấn luyện Linear Probe...")
for epoch in range(epochs):
    classifier.train()
    running_loss = 0.0
    for inputs, labels in train_features_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = classifier(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Chu kỳ {epoch+1}/{epochs} - Loss: {running_loss/len(train_features_loader):.4f}")

# 5. Đánh giá kết quả Few-shot
classifier.eval()
with torch.no_grad():
    test_outputs = classifier(X_test.to(device))
    _, test_preds = test_outputs.max(dim=1)

fs_acc = accuracy_score(y_test.numpy(), test_preds.cpu().numpy())
fs_f1 = f1_score(y_test.numpy(), test_preds.cpu().numpy(), average='macro')

print("\n" + "="*40)
print("KẾT QUẢ ĐÁNH GIÁ FEW-SHOT ĐA PHƯƠNG THỨC")
print("="*40)
print(f"Độ chính xác: {fs_acc*100:.2f}%")
print(f"Điểm F1 Macro: {fs_f1*100:.2f}%")

# Analysis

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Xây dựng lại mạng phân loại với kích thước đầu vào động
class DynamicClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=200):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

def train_and_eval_scenario(X_tr, y_tr, X_te, y_te, input_dim, device="cuda"):
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

    model = DynamicClassifier(input_dim=input_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)

    # Huấn luyện siêu tốc 50 chu kỳ
    model.train()
    for epoch in range(50):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()

    # Đánh giá
    model.eval()
    with torch.no_grad():
        preds = model(X_te.to(device)).argmax(dim=1).cpu()

    return f1_score(y_te.numpy(), preds.numpy(), average='macro')

# 1. Cấu hình 3 kịch bản cắt bỏ
scenarios = {
    "Chỉ dùng Hình ảnh": {"train": X_train[:, :512], "test": X_test[:, :512], "dim": 512},
    "Chỉ dùng Văn bản": {"train": X_train[:, 512:], "test": X_test[:, 512:], "dim": 512},
    "Đa phương thức": {"train": X_train, "test": X_test, "dim": 1024}
}

# 2. Tiến hành chạy thực nghiệm
results_f1 = {}
print("BẮT ĐẦU PHÂN TÍCH CẮT BỎ...\n")

for name, data in scenarios.items():
    print(f"Đang huấn luyện mô hình: {name}...")
    f1 = train_and_eval_scenario(
        data["train"], y_train,
        data["test"], y_test,
        data["dim"], device
    )
    results_f1[name] = f1
    print(f"-> Điểm F1 Macro: {f1*100:.2f}%\n")

# 3. Trực quan hóa kết quả
plt.figure(figsize=(9, 6))
ax = sns.barplot(x=list(results_f1.keys()), y=[v * 100 for v in results_f1.values()], palette="viridis")

plt.title("Đánh giá mức độ đóng góp của từng luồng thông tin", fontsize=14, fontweight='bold')
plt.ylabel("Điểm F1 Macro (%)", fontsize=12)
plt.ylim(0, max(results_f1.values()) * 100 + 15)

# Hiển thị số liệu trên đỉnh cột
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 8),
                textcoords='offset points',
                fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Định nghĩa các mức k (số mẫu trên mỗi lớp) cần thí nghiệm
k_values = [1, 3, 5, 10, 16]
few_shot_results = {}

print("BẮT ĐẦU THÍ NGHIỆM CÁC MỨC FEW-SHOT...")
print(f"Các mức k sẽ thử nghiệm: {k_values}\n")

for k in k_values:
    print(f"{'-'*40}")
    print(f"Đang chạy kịch bản Few-shot với k = {k} mẫu/lớp")
    print(f"{'-'*40}")

    # 2. Khởi tạo tập dữ liệu với mức k tương ứng
    fs_dataset = CUBMultimodalDataset(
        root_dir=cub_root_dir, # Biến thư mục gốc từ các bước trước
        split='train',
        processor=processor,
        k_shots=k,
        seed=42 # Giữ nguyên seed để kết quả ổn định qua các lần chạy
    )
    fs_loader = DataLoader(fs_dataset, batch_size=64, shuffle=True)

    # 3. Trích xuất đặc trưng cho tập huấn luyện thu gọn này
    X_train_k, y_train_k = extract_multimodal_features(fs_loader, clip_model, device)

    # 4. Huấn luyện mạng tuyến tính siêu tốc và lấy điểm F1
    # Tận dụng lại hàm train_and_eval_scenario đã định nghĩa ở phần cắt bỏ
    f1 = train_and_eval_scenario(
        X_train_k, y_train_k,
        X_test, y_test,
        input_dim=1024, # 1024 vì ta dùng Đa phương thức (512 Ảnh + 512 Chữ)
        device=device
    )

    few_shot_results[k] = f1
    print(f"\n=> ĐIỂM F1 MACRO ĐẠT ĐƯỢC KHI k={k}: {f1*100:.2f}%\n")

# 5. Vẽ đường cong Few-Shot Learning
plt.figure(figsize=(10, 6))

# Tách keys và values ra để vẽ
x_points = list(few_shot_results.keys())
y_points = [v * 100 for v in few_shot_results.values()]

# Vẽ đường cơ sở Zero-shot (50.69%) để làm mốc so sánh
zero_shot_baseline = 50.69
plt.axhline(y=zero_shot_baseline, color='red', linestyle='--', label=f'Zero-shot Baseline ({zero_shot_baseline}%)')

# Vẽ đường cong Few-shot
sns.lineplot(x=x_points, y=y_points, marker='o', markersize=8, linewidth=2.5, color='b', label='Few-shot (Linear Probe)')

# Điền số liệu lên từng điểm trên đồ thị
for x, y in zip(x_points, y_points):
    plt.annotate(f"{y:.1f}%",
                 (x, y),
                 textcoords="offset points",
                 xytext=(0, 10),
                 ha='center',
                 fontweight='bold')

plt.title('Đường cong học ít mẫu (Few-shot Learning Curve) trên tập CUB-200', fontsize=14, fontweight='bold')
plt.xlabel('Số lượng mẫu huấn luyện mỗi lớp (k-shots)', fontsize=12)
plt.ylabel('Điểm F1 Macro (%)', fontsize=12)
plt.xticks(x_points) # Hiển thị đúng các mốc k trên trục X
plt.ylim(zero_shot_baseline - 5, max(y_points) + 10)
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='lower right')
plt.tight_layout()

plt.show()

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score

# 1. Định nghĩa mạng phân loại với trọng số khởi tạo từ văn bản
class TextInitializedClassifier(nn.Module):
    def __init__(self, zero_shot_text_features):
        super().__init__()
        # Kích thước đầu vào là 512 (chỉ dùng ảnh) và đầu ra là 200 lớp
        self.classifier = nn.Linear(512, 200, bias=False)

        # Điểm mấu chốt: Thay vì ngẫu nhiên, gán ma trận văn bản làm trọng số
        # zero_shot_text_features có kích thước (200, 512)
        self.classifier.weight.data = zero_shot_text_features.clone().float()

    def forward(self, x):
        # Nhân thêm hệ số nhiệt độ tương tự mạng CLIP gốc giúp hội tụ nhanh
        return self.classifier(x) * 100.0

def train_and_eval_fewshot_improved(X_tr, y_tr, X_te, y_te, text_weights, device="cuda"):
    # Chỉ lấy phần đặc trưng hình ảnh (512 chiều đầu tiên) để tránh nhiễu
    X_tr_img = X_tr[:, :512]
    X_te_img = X_te[:, :512]

    train_loader = DataLoader(TensorDataset(X_tr_img, y_tr), batch_size=32, shuffle=True)

    model = TextInitializedClassifier(text_weights).to(device)
    criterion = nn.CrossEntropyLoss()

    # Dùng tốc độ học cực nhỏ để không phá vỡ trọng số ngôn ngữ đã rất tốt
    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)

    model.train()
    for epoch in range(30): # Giảm số chu kỳ vì mô hình đã bắt đầu ở vạch đích
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_te_img.to(device)).argmax(dim=1).cpu()

    return f1_score(y_te.numpy(), preds.numpy(), average='macro')

# 2. Chạy lại vòng lặp thí nghiệm
print("BẮT ĐẦU THÍ NGHIỆM FEW-SHOT VỚI TRỌNG SỐ NGÔN NGỮ...\n")
few_shot_results_improved = {}
k_values = [1, 3, 5, 10, 16]

# Biến text_features đã được sinh ra từ hàm Zero-shot được tối ưu trước đó
# Giả định bạn vẫn còn giữ biến text_features trong bộ nhớ Colab
for k in k_values:
    print(f"Đang huấn luyện mức k = {k}...")

    fs_dataset = CUBMultimodalDataset(
        root_dir=cub_root_dir,
        split='train', processor=processor, k_shots=k, seed=42
    )
    fs_loader = DataLoader(fs_dataset, batch_size=64, shuffle=True)

    X_train_k, y_train_k = extract_multimodal_features(fs_loader, clip_model, device)

    # Gọi hàm huấn luyện mới
    f1 = train_and_eval_fewshot_improved(
        X_train_k, y_train_k,
        X_test, y_test,
        text_weights=text_features, # Truyền ma trận văn bản vào đây
        device=device
    )

    few_shot_results_improved[k] = f1
    print(f"-> ĐIỂM F1 MACRO ĐẠT ĐƯỢC: {f1*100:.2f}%\n")